<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 8


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Cоздать базовый класс CreditCard в C#, который будет представлять информацию
о кредитных картах. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма

Требования к базовому классу CreditCard:
• Атрибуты: Номер карты (CardNumber), Холдера (HolderName), Срок
действия (ExpiryDate).
• Методы:
o GetInfo(): метод для получения информации о кредитной карте в виде
строки.
o Pay(): метод для оплаты покупки с использованием карты.
o CheckBalance(): метод для проверки баланса на карте.
Требования к производным классам:
1. GoldCard (GoldCreditCard): Должен содержать дополнительные атрибуты,
такие как Бесплатные бонусные мили (BonusMiles). Метод Pay() должен быть
переопределен для добавления информации о получении бонусных миль
при оплате покупки.
2. PremiumCard (PremiumCreditCard): Должен содержать дополнительные
атрибуты, такие как Ассистент поддержки (SupportAssistant).
Метод CheckBalance() должен быть переопределен для предоставления
возможности обратиться за помощью к ассистенту поддержки в случае
проблем с балансом.
3. CorporateCard (CorporateCreditCard) (если требуется третий класс): Должен
содержать дополнительные атрибуты, такие как Компания (Company).
Метод GetInfo() должен быть переопределен для включения информации о
компании в описании карты.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;

public interface INotificationService
{
    void SendMessage(string message);
}

public class ConsoleNotificationService : INotificationService
{
    public void SendMessage(string message)
    {
        Console.WriteLine($"Уведомление: {message}");
    }
}

public interface IReportable
{
    void GenerateReport();
}

public interface IBonusProgram
{
    decimal BonusBalance { get; set; }
    void AddBonus(decimal amount);
    void RedeemBonus(decimal amount);
}

public class CreditCard
{
    protected readonly INotificationService _notificationService;
    public string CardNumber { get; set; }
    public string HolderName { get; set; }
    public DateTime ExpiryDate { get; set; }
    public string Currency { get; set; } = "RUB";
    public string PinCode { get; private set; }
    public bool IsBlocked { get; set; } = false;

    private decimal _balance;
    public decimal Balance
    {
        get => _balance;
        set
        {
            if (value < 0)
                _notificationService.SendMessage("Ошибка: баланс не может быть отрицательным.");
            else
                _balance = value;
        }
    }

    public CreditCard(string cardNumber, string holderName, DateTime expiryDate, INotificationService notificationService, string currency = "RUB", string pinCode = "0000", decimal initialBalance = 0)
    {
        CardNumber = cardNumber;
        HolderName = holderName;
        ExpiryDate = expiryDate;
        _notificationService = notificationService ?? throw new ArgumentNullException(nameof(notificationService));
        Currency = currency;
        PinCode = pinCode;
        Balance = initialBalance;
    }

    public virtual string GetInfo()
    {
        return $"Номер карты: {CardNumber}, Холдер: {HolderName}, Срок действия: {ExpiryDate:MM.yyyy}, Баланс: {Balance:C} {Currency}, Статус: {(IsBlocked ? "Заблокирована" : "Активна")}";
    }

    public virtual void Pay(decimal amount)
    {
        if (IsBlocked) 
        {    
            _notificationService.SendMessage($"Ошибка: Карта {CardNumber} заблокирована. Оплата невозможна."); 
            return;
        }

        if (Balance >= amount)
        {
            Balance -= amount;
            _notificationService.SendMessage($"Оплата покупки на сумму {amount:C} с карты {CardNumber}. Остаток: {Balance:C}");
        }
        else
        {
            _notificationService.SendMessage($"Недостаточно средств на карте {CardNumber} для оплаты {amount:C}");
        }
    }

    public virtual void CheckBalance()
    {
        _notificationService.SendMessage($"Проверка баланса карты {CardNumber}: {Balance:C} {Currency}");
    }

    public bool ChangePin(string oldPin, string newPin)
    {
        if (PinCode == oldPin)
        {
            PinCode = newPin;
            _notificationService.SendMessage($"ПИН-код для карты {CardNumber} успешно изменен.");
            return true;
        }
        _notificationService.SendMessage("Неверный текущий ПИН-код.");
        return false;
    }

    public void BlockCard()
    {
        IsBlocked = true;
        _notificationService.SendMessage($"Карта {CardNumber} заблокирована.");
    }

    public void UnBlockCard()
    {
        IsBlocked = false;
        _notificationService.SendMessage($"Карта {CardNumber} разблокирована.");
    }

    public bool TransferTo(CreditCard targetCard, decimal amount)
    {
        if (IsBlocked || targetCard == null || targetCard.IsBlocked)
        {
            _notificationService.SendMessage($"[Перевод] Операция невозможна: одна из карт заблокирована или не существует.");
            return false;
        }

        if (Balance >= amount)
        {
            Balance -= amount;
            targetCard.Balance += amount;
            _notificationService.SendMessage($"[Перевод] Успешно переведено {amount:C} с карты {CardNumber} на карту {targetCard.CardNumber}.");
            return true;
        }

        _notificationService.SendMessage($"[Перевод] Недостаточно средств на карте {CardNumber} для перевода {amount:C}.");
        return false;
    }
}

public class GoldCard : CreditCard, IBonusProgram
{
    public int BonusMiles { get; set; }
    public decimal CashbackRate { get; set; } = 0.05m;
    public decimal SpecialPrivilegeDiscount { get; set; } = 0.1m;

    private decimal _bonusBalance;
    decimal IBonusProgram.BonusBalance
    {
        get => _bonusBalance;
        set => _bonusBalance = value;
    }

    public GoldCard(string cardNumber, string holderName, DateTime expiryDate, INotificationService notificationService, decimal initialBalance = 0, int bonusMiles = 0)
        : base(cardNumber, holderName, expiryDate, notificationService, "RUB", "0000", initialBalance)
    {
        BonusMiles = bonusMiles;
    }

    public override void Pay(decimal amount)
    {
        if (Balance >= amount && !IsBlocked)
        {
            base.Pay(amount);
            int earnedMiles = (int)(amount / 10);
            BonusMiles += earnedMiles;
            
            ((IBonusProgram)this).AddBonus(amount * CashbackRate);
            _notificationService.SendMessage($"За покупку начислено {earnedMiles} бонусных миль. Всего миль: {BonusMiles}");
        }
        else
        {
            _notificationService.SendMessage($"Не удалось совершить покупку по Gold-карте {CardNumber}.");
        }
    }

    void IBonusProgram.AddBonus(decimal amount)
    {
        _bonusBalance += amount;
        _notificationService.SendMessage($"[Бонусная программа] Начислено {amount:C} кэшбэка.");
    }

    void IBonusProgram.RedeemBonus(decimal amount)
    {
        if (_bonusBalance >= amount)
        {
            _bonusBalance -= amount;
            Balance += amount;
            _notificationService.SendMessage($"[Бонусная программа] Списано {amount:C} с бонусного счета на баланс карты.");
        }
    }
}

public class PremiumCard : CreditCard, IReportable
{
    public string SupportAssistant { get; set; }
    public bool IsInsured { get; set; }
    public decimal CreditLimit { get; set; } = 300000;
    public string ConciergeServicePhone { get; set; } = "+7-800-200-00";

    public PremiumCard(string cardNumber, string holderName, DateTime expiryDate, INotificationService notificationService, decimal initialBalance = 0, string supportAssistant = "Персональный менеджер", bool isInsured = true)
        : base(cardNumber, holderName, expiryDate, notificationService, "RUB", "0000", initialBalance)
    {
        SupportAssistant = supportAssistant;
        IsInsured = isInsured;
    }

    public override void CheckBalance()
    {
        base.CheckBalance();
        if (Balance < 1000)
        {
            _notificationService.SendMessage($"[Премиум] Баланс ниже лимита. Ассистент {SupportAssistant} готов помочь с пополнением.");
        }
        else
        {
            _notificationService.SendMessage($"[Премиум] Ваш личный ассистент поддержки — {SupportAssistant}.");
        }
    }

    public void DirectCallSupport()
    {
        _notificationService.SendMessage($"Соединяем с консьерж-сервисом по номеру {ConciergeServicePhone}...");
    }

    void IReportable.GenerateReport()
    {
        Console.WriteLine($"\n--- Премиальный отчет по карте {CardNumber} ---");
        Console.WriteLine($"Владелец: {HolderName}");
        Console.WriteLine($"Ассистент: {SupportAssistant}");
        Console.WriteLine($"Кредитный лимит: {CreditLimit:C}");
        Console.WriteLine($"Страховка: {(IsInsured ? "Активна" : "Не активна")}");
        Console.WriteLine($"Консьерж-телефон: {ConciergeServicePhone}\n");
    }
}

public class CorporateCard : CreditCard
{
    public string Company { get; set; }
    public int EmployeeCount { get; set; } = 50;
    public decimal DailyExpenseLimit { get; set; } = 150000;

    public CorporateCard(string cardNumber, string holderName, DateTime expiryDate, string company, INotificationService notificationService, decimal initialBalance = 0)
        : base(cardNumber, holderName, expiryDate, notificationService, "RUB", "0000", initialBalance)
    {
        Company = company;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Компания: {Company}, Сотрудников: {EmployeeCount}";
    }

    public void ApproveCorporateExpense(decimal amount)
    {
        _notificationService.SendMessage($"[Корпоративный отдел] Расход {amount:C} для компании {Company} успешно согласован.");
    }

    public void TransferBonusTo(GoldCard targetGoldCard, int miles)
    {
        targetGoldCard.BonusMiles += miles;
        _notificationService.SendMessage($"Компания {Company} перевела {miles} бонусных миль на карту {targetGoldCard.CardNumber}.");
    }
}

INotificationService notifier = new ConsoleNotificationService();

GoldCard goldCard = new GoldCard("1234-5678-9012-3456", "Марк Серков", new DateTime(2029, 12, 1), notifier, 5000, 200);
PremiumCard premiumCard = new PremiumCard("1111-3333-5555-8888", "Андрей Зелинский", new DateTime(2029, 12, 1), notifier, 800, "Анна", true);
CorporateCard corporateCard = new CorporateCard("9999-9999-9999-9998", "Дмитрий Ицков", new DateTime(2033, 6, 1), "ООО RRepiv", notifier, 20000);

Console.WriteLine(goldCard.GetInfo());
Console.WriteLine(premiumCard.GetInfo());
Console.WriteLine(corporateCard.GetInfo());

Console.WriteLine();
goldCard.Pay(1500);

IBonusProgram bonusProgram = goldCard;
bonusProgram.RedeemBonus(50);

Console.WriteLine();
premiumCard.CheckBalance();

IReportable reportable = premiumCard;
reportable.GenerateReport();

Console.WriteLine();
corporateCard.ApproveCorporateExpense(45000);

Console.WriteLine();
premiumCard.DirectCallSupport();
corporateCard.TransferTo(goldCard, 3000);
goldCard.TransferTo(premiumCard, 1000);
corporateCard.TransferBonusTo(goldCard, 500);

Console.WriteLine();
corporateCard.ChangePin("0000", "4321");
corporateCard.BlockCard();
corporateCard.Pay(500);
corporateCard.UnBlockCard();

Console.WriteLine();
goldCard.CheckBalance();
premiumCard.CheckBalance();
corporateCard.CheckBalance();

Номер карты: 1234-5678-9012-3456, Холдер: Марк Серков, Срок действия: 12.2029, Баланс: 5 000,00 ₽ RUB, Статус: Активна
Номер карты: 1111-3333-5555-8888, Холдер: Андрей Зелинский, Срок действия: 12.2029, Баланс: 800,00 ₽ RUB, Статус: Активна
Номер карты: 9999-9999-9999-9998, Холдер: Дмитрий Ицков, Срок действия: 06.2033, Баланс: 20 000,00 ₽ RUB, Статус: Активна, Компания: ООО RRepiv, Сотрудников: 50

Уведомление: Оплата покупки на сумму 1 500,00 ₽ с карты 1234-5678-9012-3456. Остаток: 3 500,00 ₽
Уведомление: [Бонусная программа] Начислено 75,00 ₽ кэшбэка.
Уведомление: За покупку начислено 150 бонусных миль. Всего миль: 350
Уведомление: [Бонусная программа] Списано 50,00 ₽ с бонусного счета на баланс карты.

Уведомление: Проверка баланса карты 1111-3333-5555-8888: 800,00 ₽ RUB
Уведомление: [Премиум] Баланс ниже лимита. Ассистент Анна готов помочь с пополнением.

--- Премиальный отчет по карте 1111-3333-5555-8888 ---
Владелец: Андрей Зелинский
Ассистент: Анна
Кредитный лимит: 300 000,00 ₽